Using the BIS data portal which keeps an updated version of all central bank speeches, I remove all non relevant ones and retain simply all the Banque de france speechs and save it in a respective folder, all ecb speeches and save it in a respective folder, and finally all FED speechs and save it in a respective folder. 

Packages

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

Path and original dataframe

In [2]:
central_bank_speeches_path = Path("/Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Data Folders/")
central_bank_speeches = pd.read_csv(central_bank_speeches_path / "speeches.csv")

In [3]:
print(central_bank_speeches.columns)
print(central_bank_speeches.shape)

Index(['url', 'title', 'description', 'date', 'text', 'author'], dtype='object')
(20728, 6)


In [4]:
print(central_bank_speeches['description'].dropna().unique())

["Speech by the Deputy Governor of the People's Bank of China, Mr. Chen Yuan, at the Bank of England Seminar held in London on 10/9/96."
 "Speech by the Governor of the People's Bank of China, Mr. Dai Xianglong, at the Hong Kong Association of Banks' Dinner on 13/11/96."
 "Speech by the Governor of the People's Bank of China, Mr. Dai Xianglong, at the luncheon hosted by the Hong Kong Financial Secretary in Washington on 30/9/96."
 ...
 'Remarks by Mr Frank Elderson, Member of the Executive Board of the European Central Bank and Vice-Chair of the Supervisory Board of the European Central Bank and Alejandra Kindelán, Chair of the Spanish Banking Association, at the "Competitiveness for growth" event organised by the Spanish Banking Association, Madrid, 19 June 2026.'
 'Text of the 10th Bundesbank IAW Lecture by Dr Joachim Nagel, President of the Deutsche Bundesbank, at the Institute for Applied Economic Research (IAW), University of Tübingen, Tübingen, 9 July 2027.'
 "Speech by Dr Fritzi

In [8]:
import re
from pathlib import Path

print("columns:", central_bank_speeches.columns.tolist())

def get_institution_segment(desc):
    """Extract the 'role + institution' portion of the description,
    i.e. everything before the speaker's name/title marker."""
    if not isinstance(desc, str):
        return ''
    m = re.search(r'^(.*?),\s*(?:Mr\.|Ms\.|Mrs\.|Dr\.|Prof\.|H\.E\.|M\.|Mme\.)', desc)
    return m.group(1) if m else desc  # fallback: use whole description (e.g. press releases with no name)

# Build a dedicated column once, reuse for all filters
central_bank_speeches['institution_segment'] = central_bank_speeches['description'].apply(get_institution_segment)

def rows_for_keywords(df, keywords, col='institution_segment'):
    pattern = '|'.join(keywords)  # keywords already contain \b boundaries where needed
    mask = df[col].fillna('').str.contains(pattern, case=False, regex=True)
    return df[mask]

# Cleaned-up keyword lists — specific phrases only, no bare substrings
fed_keywords = [r'\bfederal reserve\b', r'\bboard of governors\b']
ecb_keywords = [r'\beuropean central bank\b', r'\bECB\b']
bdf_keywords = [r'\bbanque de france\b', r'\bbank of france\b']

fed = rows_for_keywords(central_bank_speeches, fed_keywords)
ecb = rows_for_keywords(central_bank_speeches, ecb_keywords)
bdf = rows_for_keywords(central_bank_speeches, bdf_keywords)

print("FED hits:", fed.shape[0])
print("ECB hits:", ecb.shape[0])
print("Banque de France hits:", bdf.shape[0])

for name, df in [('FED', fed), ('ECB', ecb), ('Banque de France', bdf)]:
    print(name, "sample descriptions:")
    print(df['description'].dropna().unique()[:5])

out_dir = Path("/Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Data Folders/filtered_speeches")
out_dir.mkdir(parents=True, exist_ok=True)
fed.drop(columns='institution_segment').to_csv(out_dir / "fed_speeches.csv", index=False)
ecb.drop(columns='institution_segment').to_csv(out_dir / "ecb_speeches.csv", index=False)
bdf.drop(columns='institution_segment').to_csv(out_dir / "banque_de_france_speeches.csv", index=False)

columns: ['url', 'title', 'description', 'date', 'text', 'author', 'institution_segment']
FED hits: 2671
ECB hits: 2821
Banque de France hits: 556
FED sample descriptions:
['Remarks by the Vice Chairman of the Board of Governors of the US Federal Reserve System, Ms. Alice M. Rivlin, at the The Brookings Institution National Issues Forum in Washington on 19/12/96.'
 'Remarks by Mr. Laurence H. Meyer, a member of the Board of Governors of the US Federal Reserve System, at the Forecasters Club of New York on 24/4/97.'
 "Remarks by Mr. Edward W. Kelley, Jr., a member of the Board of Governors of the US Federal Reserve System, before the Professional Banker's Association, Washington, D.C. on 15/12/97."
 'Remarks by the Chairman of the Board of the U S Federal Reserve System, Mr. Alan Greenspan, at the Economic Club of New York in New York City, on 02/12/97.'
 'Remarks by the President of the Federal Reserve Bank of New York, Mr. William J. McDonough, before the Association of German Mortgag